In [13]:
import sys
import torch
from torch.utils.data import DataLoader
from torch import nn
import torch.nn.functional as F #원-핫-인코딩 모듈

import numpy as np

import torchvision.datasets as datasets
from torchvision.transforms import ToTensor

mnist_train = datasets.MNIST(root='./data', download=True, train=True, transform=ToTensor() )
mnist_test = datasets.MNIST(root='./data', download=True, train=False, transform=ToTensor() )

train_dataloader = DataLoader( mnist_train, batch_size=32, shuffle=True )
test_dataloader = DataLoader( mnist_test, batch_size=32, shuffle=True )

# 모델 설정
'''
입력 784: 각 이미지의 픽셀 수 (28×28=784)
출력 100: 히든 레이어의 뉴런 개수 (임의로 설정한 값)
너무 적으면: 정보 손실
너무 많으면: 과적합, 계산 비용 증가
'''
model = nn.Sequential(
    nn.Linear( 784, 50 ),
    nn.ReLU(),
    nn.Linear( 50, 50),
    nn.ReLU(),
    nn.Linear( 50, 10 ) # 출력값 10개 뉴런
)

# loss_fn = torch.nn.BCEWithLogitsLoss()
loss_fn = torch.nn.CrossEntropyLoss() # 소프트맥스 손실함수 계산을 위해 교차 엔트로피 설정
optimizer = torch.optim.Adam( model.parameters(), lr=0.001 )

train_losses = []
val_losses = []

# 한 번의 반복(iteration)에서 처리되는 데이터:
for i in range( 0, 10 ):
    model.train()
    loss_sum = 0
    for X,y in train_dataloader:
        
        X = X.reshape((-1, 784 )) # [32, 784] - 32개 이미지를 각각 784픽셀로
        y = F.one_hot(y, num_classes=10).type(torch.float32)

        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        loss_sum+=loss.item()
       
    train_losses.append( loss_sum / len(train_dataloader) )

print( train_losses )

torch.save( model.state_dict(), 'mnist-model.pth' )

# 0값을 추출함 > but, 원-핫 인코딩에 대응하지 못함
# model.eval()
# with torch.no_grad():
#     accurate = 0
#     total = 0
#     for X, y in test_dataloader:
#         X = X.reshape((-1, 784))
#         outputs = nn.functional.softmax( model(X), dim=1 )
#         correct_pred = ( y == outputs.max(dim=1).indices )
#         total+=correct_pred.size(0)
#         accurate += correct_pred.type( torch.int ).sum().item()
    
#     print( accurate / total )









[0.3500725252896547, 0.1608107318043709, 0.1142977222914497, 0.09085831341122587, 0.076527034565527, 0.06528630676784863, 0.05650283870473504, 0.049293055472522974, 0.041549574244728625, 0.0387805637860011]
